# Hyperparameter Tuning the Winning Architecture

In Notebook 1, our baseline benchmark identified the strongest model family. 
Now, we optimize that winner using systematic hyperparameter search.

### Core Concepts Covered:
1. **GridSearchCV vs. RandomizedSearchCV**: Exhaustive search vs. budget-controlled random sampling.
2. **Pipeline Parameter Syntax**: Accessing nested steps using the `<step_name>__<param_name>` convention.
3. **Evaluating Search Results Numerically**: Inspecting the CV results table without plotting.
4. **Final Model Lockdown**: Verifying the tuned pipeline on the locked test set.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

np.random.seed(42)

# 1. Synthesize the same dataset structure
X_raw, y_raw = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    weights=[0.80, 0.20],
    random_state=42
)

X = pd.DataFrame(X_raw, columns=[f"feat_{i}" for i in range(20)])
y = pd.Series(y_raw, name="target")

# 2. Train/Test split (locked holdout set)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Base Pipeline using the winner from Notebook 1
pipeline = Pipeline([
    ('classifier', HistGradientBoostingClassifier(random_state=42))
])

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (4000, 20)
X_test shape:  (1000, 20)


---
## Step 1: Define the Hyperparameter Search Space

When tuning through a Scikit-Learn `Pipeline`, specify parameters using:
`classifier__<parameter_name>`

### Key Knobs for HistGradientBoosting:
* `learning_rate`: Step size shrinkage used to prevent overfitting ($0.01$ to $0.2$).
* `max_iter`: Maximum number of boosting trees to build.
* `max_depth`: Maximum depth of each individual tree (controls feature interactions).
* `min_samples_leaf`: Minimum samples required in a leaf (smooths out noisy splits).
* `l2_regularization`: L2 penalty on leaf values.

In [3]:
param_distributions = {
    'classifier__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.15],
    'classifier__max_iter': [100, 150, 200, 250],
    'classifier__max_depth': [3, 5, 8, None],
    'classifier__min_samples_leaf': [10, 20, 40, 80],
    'classifier__l2_regularization': [0.0, 0.1, 1.0, 5.0]
}

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Randomized search evaluates 25 random combinations (budget-friendly)
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=25,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    return_train_score=False
)

print("Starting Randomized Search...")
search.fit(X_train, y_train)
print("Search Complete.")

Starting Randomized Search...
Search Complete.


---
## Step 2: Inspecting Search Results Numerically

We pull the top 5 configurations directly from `cv_results_` to verify stability and see which hyperparameter ranges delivered the best performance.

In [6]:
cv_results = pd.DataFrame(search.cv_results_)

# Filter down to essential columns for clean tabular inspection
summary_cols = [
    'params',
    'mean_test_score',
    'std_test_score',
    'rank_test_score',
    'mean_fit_time'
]

top_configs = (cv_results[summary_cols].sort_values(by='rank_test_score').head(5).reset_index(drop=True))

# Unpack parameters dictionary into clean columns
params_expanded = pd.json_normalize(top_configs['params'])
top_configs_clean = pd.concat([params_expanded, top_configs[['mean_test_score', 'std_test_score', 'mean_fit_time']]], axis=1)

print("=== TOP 5 HYPERPARAMETER CONFIGURATIONS ===")
display(top_configs_clean)

print(f"\nBest CV ROC-AUC: {search.best_score_:.4f}")
print("Best Parameter Set:")
for param, val in search.best_params_.items():
    print(f"  {param}: {val}")

=== TOP 5 HYPERPARAMETER CONFIGURATIONS ===


,classifier__min_samples_leaf,classifier__max_iter,classifier__max_depth,classifier__learning_rate,classifier__l2_regularization,mean_test_score,std_test_score,mean_fit_time
0,10,250,NaN,0.10,0.1,0.973942,0.006903,1.572338
1,80,250,8.0,0.10,0.0,0.972946,0.004463,0.966303
2,40,150,NaN,0.15,0.1,0.972559,0.006643,0.761786
3,80,150,8.0,0.10,0.0,0.972143,0.004727,0.451922
4,10,200,5.0,0.15,5.0,0.972044,0.006728,0.995022



Best CV ROC-AUC: 0.9739
Best Parameter Set:
  classifier__min_samples_leaf: 10
  classifier__max_iter: 250
  classifier__max_depth: None
  classifier__learning_rate: 0.1
  classifier__l2_regularization: 0.1


---
## Step 3: Final Audit on the Test Set

`RandomizedSearchCV` automatically refits the best configuration on the entire `X_train`.
We now test this optimized model once on our untouched `X_test`.

In [5]:
best_model = search.best_estimator_

# Predict probabilities on test set
y_test_probs = best_model.predict_proba(X_test)[:, 1]
y_test_preds = best_model.predict(X_test)

test_roc = roc_auc_score(y_test, y_test_probs)
test_pr = average_precision_score(y_test, y_test_probs)

print("=== FINAL TEST SET PERFORMANCE (TUNED MODEL) ===")
print(f"Test ROC-AUC: {test_roc:.4f}")
print(f"Test PR-AUC:  {test_pr:.4f}\n")

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_test_preds, target_names=['Class 0', 'Class 1']))

=== FINAL TEST SET PERFORMANCE (TUNED MODEL) ===
Test ROC-AUC: 0.9784
Test PR-AUC:  0.9548

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     Class 0       0.96      0.99      0.97       797
     Class 1       0.94      0.84      0.89       203

    accuracy                           0.96      1000
   macro avg       0.95      0.91      0.93      1000
weighted avg       0.96      0.96      0.96      1000

